In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "state-spaces/mamba-130m-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
)
model.to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-23): 24 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLUActivation()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [11]:
prompt = "Explain photosynthesis in one paragraph."
encoded = tokenizer([prompt], return_tensors="pt")
input_ids = encoded["input_ids"].to(DEVICE)

with torch.inference_mode():
    output = model(input_ids=input_ids, use_cache=True, return_dict=True)

cache_obj = getattr(output, "cache_params", None)
if cache_obj is None:
    cache_obj = getattr(output, "past_key_values", None)

type(cache_obj)


transformers.cache_utils.DynamicCache

In [12]:
from state_spectrum_sweep.run_experiment_mamba import extract_cache_tensors

views = extract_cache_tensors(cache_obj, subset="all")
print(len(views))
print(views[0].path.name, views[0].tensor.shape, views[0].tensor.dtype, views[0].tensor.device)


48
layers.0.conv_states torch.Size([1, 1536, 4]) torch.bfloat16 cuda:0


In [13]:
ssm_states = extract_cache_tensors(cache_obj, subset="ssm")
conv_states = extract_cache_tensors(cache_obj, subset="conv")

In [14]:
state_tensors = {
    view.path.name: view.tensor.detach().float().cpu().clone()
    for view in views
}


In [23]:
x = state_tensors["layers.23.recurrent_states"]
x.shape, x.layout, x

(torch.Size([1, 1536, 16]),
 torch.strided,
 tensor([[[ 0.0092, -0.0500,  0.0044,  ...,  0.0255, -0.0693, -0.0811],
          [-0.0064,  0.0068, -0.0030,  ..., -0.0222,  0.0520,  0.0708],
          [-0.0669,  0.0684, -0.0084,  ..., -0.0454,  0.1289,  0.1582],
          ...,
          [-0.0069,  0.0093, -0.0028,  ..., -0.0140,  0.0369,  0.0491],
          [-0.1108,  0.0527, -0.0167,  ..., -0.0894,  0.2129,  0.2852],
          [-0.0014,  0.0021, -0.0007,  ...,  0.0063,  0.0081,  0.0123]]]))

In [9]:
def sparsity_stats(x, tol=1e-6):
    x = x.detach().float().cpu()
    return {
        "shape": tuple(x.shape),
        "zero_frac": float((x == 0).float().mean()),
        "near_zero_frac": float((x.abs() < tol).float().mean()),
        "mean_abs": float(x.abs().mean()),
        "max_abs": float(x.abs().max()),
    }

for name, tensor in list(state_tensors.items())[:10]:
    print(name, sparsity_stats(tensor))


layers.0.conv_states {'shape': (1, 1536, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.3469269275665283, 'max_abs': 6.09375}
layers.0.recurrent_states {'shape': (1, 1536, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.0060628256760537624, 'mean_abs': 0.02432837337255478, 'max_abs': 2.125}
layers.1.conv_states {'shape': (1, 1536, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.4529827833175659, 'max_abs': 3.3125}
layers.1.recurrent_states {'shape': (1, 1536, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.0015055338153615594, 'mean_abs': 0.006815113592892885, 'max_abs': 0.328125}
layers.2.conv_states {'shape': (1, 1536, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.47100162506103516, 'max_abs': 4.21875}
layers.2.recurrent_states {'shape': (1, 1536, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.0013834635028615594, 'mean_abs': 0.0045541985891759396, 'max_abs': 0.1484375}
layers.3.conv_states {'shape': (1, 1536, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_a